# Solver capability migration — walkthrough

**Branch:** `solver-refac`  
**Goal:** capability data lives on each `Solver` subclass as `ClassVar`s. `linopy.solver_capabilities` becomes a back-compat shim.

What used to be duplicated between `solvers.py` (the classes) and `solver_capabilities.py` (`SOLVER_REGISTRY`) is now declared once, on the class itself. The registry is a lazy view over the classes.

Sections:
1. Class-level API: `Solver.features`, `Solver.supports`, `Solver.display_name`
2. Back-compat shim — every legacy import still works
3. `SOLVER_REGISTRY` is now a lazy `Mapping` view
4. Xpress GPU feature is still version-gated
5. Internal callers (`Model.solve`) use the class API directly

*Run from the repo root.*

## 1. Class-level API

Each `Solver` subclass declares `features` and `display_name` as `ClassVar`s, and inherits a `supports()` classmethod from the base class.

In [1]:
from linopy import SolverFeature  # newly re-exported at the top level
from linopy import solvers

list(SolverFeature)

[<SolverFeature.INTEGER_VARIABLES: 1>,
 <SolverFeature.QUADRATIC_OBJECTIVE: 2>,
 <SolverFeature.DIRECT_API: 3>,
 <SolverFeature.LP_FILE_NAMES: 4>,
 <SolverFeature.READ_MODEL_FROM_FILE: 5>,
 <SolverFeature.SOLUTION_FILE_NOT_NEEDED: 6>,
 <SolverFeature.GPU_ACCELERATION: 7>,
 <SolverFeature.IIS_COMPUTATION: 8>,
 <SolverFeature.SOS_CONSTRAINTS: 9>,
 <SolverFeature.SEMI_CONTINUOUS_VARIABLES: 10>,
 <SolverFeature.SOLVER_ATTRIBUTE_ACCESS: 11>]

In [2]:
print(solvers.Gurobi.display_name)
print(solvers.Gurobi.supports(SolverFeature.SOS_CONSTRAINTS))
print(solvers.Highs.supports(SolverFeature.SOS_CONSTRAINTS))
print(solvers.cuPDLPx.supports(SolverFeature.GPU_ACCELERATION))

Gurobi
True
False
True


In [3]:
# A quick capability matrix across every Solver subclass.
import pandas as pd

rows = []
for name in solvers.SolverName:
    cls = getattr(solvers, name.name)
    rows.append({"solver": name.value, **{f.name: cls.supports(f) for f in SolverFeature}})
pd.DataFrame(rows).set_index("solver")

,INTEGER_VARIABLES,QUADRATIC_OBJECTIVE,DIRECT_API,LP_FILE_NAMES,READ_MODEL_FROM_FILE,SOLUTION_FILE_NOT_NEEDED,GPU_ACCELERATION,IIS_COMPUTATION,SOS_CONSTRAINTS,SEMI_CONTINUOUS_VARIABLES,SOLVER_ATTRIBUTE_ACCESS
solver,,,,,,,,,,,
cbc,True,False,False,False,True,False,False,False,False,False,False
glpk,True,False,False,False,True,False,False,False,False,False,False
highs,True,True,True,True,True,True,False,False,False,True,False
cplex,True,True,False,True,True,False,False,False,True,True,False
gurobi,True,True,True,True,True,True,False,True,True,True,True
scip,True,True,False,True,True,True,False,False,False,False,False
xpress,True,True,False,True,True,True,False,True,False,False,False
knitro,True,True,False,True,True,True,False,False,False,False,False
mosek,True,True,True,True,True,True,False,False,False,False,False


## 2. Back-compat shim

Downstream packages (pypsa-eur, …) import from `linopy.solver_capabilities`. Every legacy symbol still resolves, but `solver_supports` / `get_solvers_with_feature` now delegate to the solver classes.

In [4]:
from linopy.solver_capabilities import (
    SOLVER_REGISTRY,
    SolverFeature as SF_shim,
    SolverInfo,
    get_available_solvers_with_feature,
    get_solvers_with_feature,
    solver_supports,
)

# Same enum object — the shim re-exports from linopy.solvers.
assert SF_shim is SolverFeature

print(solver_supports("gurobi", SolverFeature.IIS_COMPUTATION))
print(get_solvers_with_feature(SolverFeature.DIRECT_API))
print(get_available_solvers_with_feature(SolverFeature.DIRECT_API, solvers.available_solvers))

True
['highs', 'gurobi', 'mosek', 'cupdlpx']
['highs', 'gurobi']


## 3. `SOLVER_REGISTRY` is a lazy Mapping

Downstream code that does `for name in SOLVER_REGISTRY: info = SOLVER_REGISTRY[name]; ...` keeps working. `SolverInfo` is constructed on demand from the class declarations.

In [5]:
print(type(SOLVER_REGISTRY).__name__)
print(len(SOLVER_REGISTRY), "solvers known")
info = SOLVER_REGISTRY["gurobi"]
print(type(info).__name__)
print(info.name, "·", info.display_name)
print(sorted(f.name for f in info.features))

_LazyRegistry
13 solvers known
SolverInfo
gurobi · Gurobi
['DIRECT_API', 'IIS_COMPUTATION', 'INTEGER_VARIABLES', 'LP_FILE_NAMES', 'QUADRATIC_OBJECTIVE', 'READ_MODEL_FROM_FILE', 'SEMI_CONTINUOUS_VARIABLES', 'SOLUTION_FILE_NOT_NEEDED', 'SOLVER_ATTRIBUTE_ACCESS', 'SOS_CONSTRAINTS']


In [6]:
# Round-trip property: shim ↔ class API agree for every (solver, feature) pair.
mismatches = [
    (n.value, f.name)
    for n in solvers.SolverName
    for f in SolverFeature
    if solver_supports(n.value, f) != getattr(solvers, n.name).supports(f)
]
assert mismatches == [], mismatches
print("shim and class API agree on all", len(list(solvers.SolverName)) * len(list(SolverFeature)), "pairs")

shim and class API agree on all 143 pairs


## 4. Xpress GPU is still version-gated

`_xpress_supports_gpu()` is now defined in `linopy.solvers` and evaluated once at class-definition time — same semantics as before, just colocated with the class it gates.

In [7]:
from linopy.solvers import _xpress_supports_gpu

gpu = _xpress_supports_gpu()
print("xpress >= 9.8.0 installed?", gpu)
print("Xpress.supports(GPU_ACCELERATION):", solvers.Xpress.supports(SolverFeature.GPU_ACCELERATION))
assert solvers.Xpress.supports(SolverFeature.GPU_ACCELERATION) == gpu

xpress >= 9.8.0 installed? False
Xpress.supports(GPU_ACCELERATION): False


## 5. Internal callers use the class API

`Model.solve()` resolves `solver_class` once and then asks `solver_class.supports(...)` for every pre-instantiation check (quadratic objective, SOS constraints, semi-continuous vars, LP file names, solution-file-not-needed). `compute_infeasibilities()` queries `self.solver.supports(IIS_COMPUTATION)` on the already-instantiated solver.

The only remaining string-keyed call (`solver_supports(name, ...)`) lives in `linopy/variables.py`, where a constraint-add path runs before a solver instance exists. It routes through the shim — no behavior change.

In [8]:
# Quick end-to-end smoke: solve uses the new path, status is OK.
from linopy import GREATER_EQUAL, Model

m = Model()
x = m.add_variables(name="x")
y = m.add_variables(name="y")
m.add_constraints(2 * x + 6 * y, GREATER_EQUAL, 10)
m.add_constraints(4 * x + 2 * y, GREATER_EQUAL, 3)
m.add_objective(2 * y + x)
m.solve(solvers.available_solvers[0])

print("solver:", m.solver)
print("supports IIS?", m.solver.supports(SolverFeature.IIS_COMPUTATION))

Restricted license - for non-production use only - expires 2027-11-29


Read LP format model from file /tmp/linopy-problem-ft6epgcz.lp


Reading time = 0.00 seconds


obj: 2 rows, 2 columns, 4 nonzeros


Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 24.04.4 LTS")


...

Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    3.3000000e+00   0.000000e+00   0.000000e+00      0s


Solved in 0 iterations and 0.00 seconds (0.00 work units)


Optimal objective  3.300000000e+00


solver: Gurobi(name='gurobi', status='ok', io_api='lp', solver_model=loaded, env=active, objective=3.3, runtime=0.00437s)
supports IIS? True
